In [15]:
import tensorflow as tf
from tensorflow.keras.models import load_model
# Load the trained model from the HDF5 file
import pickle
import pandas as pd
import numpy as np

In [16]:
##Load the ANN trained model, scalar pickle file and the one hot encoded test data for making predictions
model = load_model('churn_model.h5')
with open('scaler.pkl', 'rb') as f:
    scaler = pickle.load(f)
with open('onehot_encoder.pkl', 'rb') as f:
    onehot_encoder_geo = pickle.load(f)
with open('label_encoder.pkl', 'rb') as f:
    label_encoder_gender = pickle.load(f)

In [17]:
##Example input data for prediction (replace with actual test data)
input_data = {
    'CreditScore': 600,
    'Geography': 'France',
    'Gender': 'Male',
    'Age': 40,
    'Tenure': 3,
    'Balance': 60000,
    'NumOfProducts': 2,
    'HasCrCard': 1,
    'IsActiveMember': 1,
    'EstimatedSalary': 50000
}

In [18]:
##One hot encode the 'Geography' feature using the loaded onehot encoder
geo_encoded = onehot_encoder_geo.transform([[input_data['Geography']]]).toarray()
geo_encoded_df = pd.DataFrame(geo_encoded, columns=onehot_encoder_geo.get_feature_names_out(['Geography']))
geo_encoded_df

c:\Users\shash\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but OneHotEncoder was fitted with feature names
  warnings.warn(


,Geography_France,Geography_Germany,Geography_Spain
0,1.0,0.0,0.0


In [19]:
input_df = pd.DataFrame([input_data])
input_df

,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary
0,600,France,Male,40,3,60000,2,1,1,50000


In [20]:
##Encode the categorical 'Gender
input_df['Gender'] = label_encoder_gender.transform(input_df['Gender'])
##Drop the original 'Geography' column as it's now encoded
input_df

,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary
0,600,France,1,40,3,60000,2,1,1,50000


In [24]:
print("Scaler features:")
print(list(scaler.feature_names_in_))

print("\nInput features:")
print(list(input_df.columns))

Scaler features:
['CreditScore', 'Gender', 'Age', 'Tenure', 'Balance', 'NumOfProducts', 'HasCrCard', 'IsActiveMember', 'EstimatedSalary', 'Geography_France', 'Geography_Germany', 'Geography_Spain', 'Geography_France', 'Geography_Germany', 'Geography_Spain']

Input features:
['CreditScore', 'Gender', 'Age', 'Tenure', 'Balance', 'NumOfProducts', 'HasCrCard', 'IsActiveMember', 'EstimatedSalary', 'Geography_France', 'Geography_Germany', 'Geography_Spain']


In [21]:
##Concatinate the encoded geography columns with the original input dataframe
input_df = pd.concat([input_df.drop("Geography", axis=1), geo_encoded_df], axis=1)

In [25]:
expected_columns = scaler.feature_names_in_

for col in expected_columns:
    if col not in input_df.columns:
        input_df[col] = input_df[col.replace('.1','')] if col.replace('.1','') in input_df.columns else 0

input_df = input_df[expected_columns]

In [26]:
##Scale the input data using the loaded scaler
scaled_input = scaler.transform(input_df)
scaled_input

array([[-0.53598516,  0.91324755,  0.10479359, -0.69539349, -0.25781119,
         0.80843615,  0.64920267,  0.97481699, -0.87683221,  1.00150113,
        -0.57946723, -0.57638802,  1.00150113, -0.57946723, -0.57638802]])

In [27]:
##Predict the probability of churn using the loaded model
prediction = model.predict(scaled_input)
print(f"Predicted probability of churn: {prediction[0][0]:.4f}")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 68ms/step
Predicted probability of churn: 0.1605


In [28]:
if prediction[0][0] > 0.5:
    print("The customer is likely to churn.")
else:
    print("The customer is unlikely to churn.")

The customer is unlikely to churn.
